# Advanced Model 1 Tuning

Extend the word-level TF-IDF and logistic-regression baseline from Notebook 02 with leakage-safe cross-validation and focused regularization analysis. The unlabeled Kaggle test set remains untouched.

In [2]:
# standard library
from math import prod
from pathlib import Path
import sys
from time import perf_counter

# third-party
import pandas as pd
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

# local
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

scripts_path = str(PROJECT_ROOT / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from imdb_dataset_helpers import clean_reviews, load_datasets

## Configuration

Keep the efficient word-level representation fixed while sweeping logistic-regression regularization.

In [3]:
RANDOM_STATE = 42
CROSS_VALIDATION_FOLDS = 5
CROSS_VALIDATION_JOBS = 4
MODEL_1_NGRAM_RANGE = (1, 2)
MODEL_1_MAX_FEATURES = 100_000
SELECTED_C = 2.0

## Load and clean the labeled data

Load a fresh copy of the training data and apply the shared cleanup. Cross-validation operates only on these labeled reviews.

In [4]:
train, _ = load_datasets()
train_clean = clean_reviews(train)

pd.Series(
    {
        "labeled_reviews": len(train_clean),
        "positive_rate": train_clean["sentiment"].mean(),
    },
    name="Cleaned training data",
)

Kaggle files already present; using: (project_root)/data/raw


labeled_reviews    24903.000000
positive_rate          0.500783
Name: Cleaned training data, dtype: float64

## Model 1: cross-validated tuning

Keep the efficient word-level TF-IDF configuration fixed while examining logistic-regression regularization more closely. The selected TF-IDF settings remain visible in the configuration cell and easy to change later. Keeping TF-IDF inside the pipeline ensures that each fold learns its vocabulary and inverse-document-frequency weights only from that fold's training reviews.

In [5]:
model_1_pipeline = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=MODEL_1_NGRAM_RANGE,
                min_df=2,
                max_features=MODEL_1_MAX_FEATURES,
                sublinear_tf=True,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1_000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

model_1_param_grid = {
    "classifier__C": [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 4.0, 6.0, 8.0],
}

cross_validation = StratifiedKFold(
    n_splits=CROSS_VALIDATION_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

model_1_search = GridSearchCV(
    estimator=model_1_pipeline,
    param_grid=model_1_param_grid,
    scoring="roc_auc",
    cv=cross_validation,
    n_jobs=CROSS_VALIDATION_JOBS,
    refit=True,
    return_train_score=True,
    verbose=2,
)

number_of_candidates = prod(
    [len(values) for values in model_1_param_grid.values()]
)
print(f"Candidates: {number_of_candidates}")
print(f"Total cross-validation fits: {number_of_candidates * CROSS_VALIDATION_FOLDS}")

Candidates: 9
Total cross-validation fits: 45


### Run the search

Fit the nine regularization candidates across five folds. The broad sweep lets us observe both training and validation behavior from strong through weak regularization. With `refit=True`, scikit-learn automatically refits the winning pipeline on all cleaned labeled reviews after cross-validation.

In [6]:
search_start = perf_counter()
model_1_search.fit(train_clean["review"], train_clean["sentiment"])
search_seconds = perf_counter() - search_start

print(f"Search time: {search_seconds:,.1f} seconds")

Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV] END ................................classifier__C=0.001; total time=  12.8s
[CV] END ................................classifier__C=0.001; total time=  12.8s
[CV] END ................................classifier__C=0.001; total time=  12.8s
[CV] END ................................classifier__C=0.001; total time=  12.8s
[CV] END ................................classifier__C=0.001; total time=  11.6s
[CV] END .................................classifier__C=0.01; total time=  11.6s
[CV] END .................................classifier__C=0.01; total time=  11.6s
[CV] END .................................classifier__C=0.01; total time=  11.6s
[CV] END .................................classifier__C=0.01; total time=  11.6s
[CV] END .................................classifier__C=0.01; total time=  11.7s
[CV] END ..................................classifier__C=0.1; total time=  12.0s
[CV] END ..................................classi

### Cross-validation results

Compare mean validation ROC-AUC and fold-to-fold variability. Training ROC-AUC and the train–validation gap make it easier to see when weaker regularization increasingly fits the training folds without a corresponding validation improvement.

In [7]:
model_1_cv_results = pd.DataFrame(model_1_search.cv_results_)
model_1_cv_results["train_validation_gap"] = (
    model_1_cv_results["mean_train_score"]
    - model_1_cv_results["mean_test_score"]
)
model_1_cv_results = (
    model_1_cv_results
    .loc[
        :,
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "train_validation_gap",
            "param_classifier__C",
            "mean_fit_time",
        ],
    ]
    .rename(
        columns={
            "rank_test_score": "rank",
            "mean_test_score": "mean_validation_roc_auc",
            "std_test_score": "validation_roc_auc_std",
            "mean_train_score": "mean_training_roc_auc",
            "param_classifier__C": "C",
            "mean_fit_time": "mean_fit_seconds",
        }
    )
    .sort_values("rank")
    .reset_index(drop=True)
)

model_1_cv_results.round(
    {
        "mean_validation_roc_auc": 5,
        "validation_roc_auc_std": 5,
        "mean_training_roc_auc": 5,
        "train_validation_gap": 5,
        "mean_fit_seconds": 2,
    }
)

,rank,mean_validation_roc_auc,validation_roc_auc_std,mean_training_roc_auc,train_validation_gap,C,mean_fit_seconds
0,1,0.96759,0.00070,0.99984,0.03225,8.000,10.95
1,2,0.96739,0.00063,0.99948,0.03209,6.000,11.31
2,3,0.96686,0.00072,0.99885,0.03199,4.000,11.22
3,4,0.96542,0.00072,0.99613,0.03072,2.000,11.99
4,5,0.96275,0.00069,0.99069,0.02794,1.000,12.16
5,6,0.95787,0.00117,0.98261,0.02474,0.500,11.02
6,7,0.94084,0.00220,0.95909,0.01825,0.100,10.62
7,8,0.91518,0.00362,0.93009,0.01491,0.010,10.28
8,9,0.91463,0.00353,0.92948,0.01485,0.001,11.16


### Choosing `C = 2`

Although `C = 8` produced the highest mean validation ROC-AUC, I selected `C = 2` as a deliberate balance between validation performance and regularization:

- Its mean validation ROC-AUC of `0.96542` is only `0.00217` below the best observed score of `0.96759`.
- Stronger regularization reduces the train–validation gap from `0.03225` at `C = 8` to `0.03072` at `C = 2`.
- Its training ROC-AUC of `0.99613` is less extreme than the nearly perfect `0.99984` reached at `C = 8`.
- The small validation-performance sacrifice is accepted in exchange for a more conservative model with lower overfitting risk.
- This is a robustness choice rather than the automatic `GridSearchCV` winner; `C = 8` remains the highest-scoring cross-validation configuration.

In [8]:
automatic_winner_summary = pd.Series(
    {
        "mean_cross_validated_roc_auc": model_1_search.best_score_,
        "ngram_range": MODEL_1_NGRAM_RANGE,
        "max_features": MODEL_1_MAX_FEATURES,
        "C": model_1_search.best_params_["classifier__C"],
        "search_seconds": search_seconds,
    },
    name="Automatic GridSearchCV winner",
)
automatic_winner_summary

mean_cross_validated_roc_auc      0.967591
ngram_range                         (1, 2)
max_features                        100000
C                                      8.0
search_seconds                  230.826346
Name: Automatic GridSearchCV winner, dtype: object

### Fit the selected robust model

The automatic ROC-AUC winner remains visible above as experimental evidence. For downstream use, explicitly create a separate pipeline with the documented `C = 2` robustness choice and fit it on all cleaned labeled reviews.

In [9]:
selected_model_1 = clone(model_1_pipeline).set_params(
    classifier__C=SELECTED_C,
)
selected_model_1.fit(
    train_clean["review"],
    train_clean["sentiment"],
)

selected_cv_result = model_1_cv_results.loc[
    model_1_cv_results["C"].eq(SELECTED_C)
].iloc[0]
selected_model_1_summary = pd.Series(
    {
        "mean_cross_validated_roc_auc": selected_cv_result["mean_validation_roc_auc"],
        "validation_roc_auc_std": selected_cv_result["validation_roc_auc_std"],
        "train_validation_gap": selected_cv_result["train_validation_gap"],
        "ngram_range": MODEL_1_NGRAM_RANGE,
        "max_features": MODEL_1_MAX_FEATURES,
        "C": SELECTED_C,
    },
    name="Selected Model 1 configuration",
)
selected_model_1_summary

mean_cross_validated_roc_auc    0.965419
validation_roc_auc_std          0.000724
train_validation_gap            0.030715
ngram_range                       (1, 2)
max_features                      100000
C                                    2.0
Name: Selected Model 1 configuration, dtype: object

### Evaluation boundary

`selected_model_1` is the explicitly chosen `C = 2` pipeline fitted on all cleaned labeled reviews.
`model_1_search.best_estimator_` remains available only as the automatic `C = 8` search winner. 
Only the deliberately selected pipeline should be used to generate probabilities for the unlabeled Kaggle test set and a competition submission.